In [0]:
from delta import configure_spark_with_delta_pip, DeltaTable
from pyspark.sql import SparkSession
from pyspark.sql.functions import expr, lit

In [0]:
builder = (SparkSession.builder
           .appName("upsert-delta-table")
           .master("spark://spark-master:7077")
           .config("spark.executor.memory", "512m")
           .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
           .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog"))

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

In [0]:
%load_ext sparksql_magic
%config SparkSql.limit=20

In [0]:
# For PySpark:
deltaTable = DeltaTable.forPath(spark, "/opt/workspace/data/delta_lake/netflix_titles")

In [0]:
deltaTable.toDF().show(5)

In [0]:
# Update director to not have nulls
deltaTable.update(
  condition = expr("director IS NULL"),
  set = { "director": lit("") })

In [0]:
%%sparksql
UPDATE delta.`/opt/workspace/data/delta_lake/netflix_titles` SET director = "" WHERE director IS NULL;

In [0]:
# Show a sample of the data
deltaTable.toDF().show(5)

In [0]:
spark.stop()